### Pip install que precisam ocorrer antes de importe de blibliotecas especificas

In [1]:

%pip install uv
!uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install xformers --index-url https://download.pytorch.org/whl/cu121

!uv pip install unsloth unsloth-zoo torchao
!uv pip install accelerate transformers trl peft bitsandbytes datasets
!uv pip install setuptools
!uv pip install pandas scikit-learn google-generativeai matplotlib ipywidgets

/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
Using Python 3.10.21 environment at: myenv
Checked 3 packages in 12ms
Using Python 3.10.21 environment at: myenv
Checked 1 package in 3ms
Using Python 3.10.21 environment at: myenv
Resolved 99 packages in 717ms                                        
Uninstalled 2 packages in 19ms
Installed 2 packages in 46ms                                
 - fsspec==2026.7.0
 + fsspec==2025.9.0
 - torchao==0.7.0+cu121
 + torchao==0.18.0
Using Python 3.10.21 environment at: myenv
Checked 6 packages in 8ms
Using Python 3.10.21 environment at: myenv
Checked 1 package in 3ms
Using Python 3.10.21 environment at: myenv
Checked 5 packages in 17ms


In [1]:
import os
import time
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import google.generativeai as genai

# Verificação de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma GPU'
print(f"Iniciando Pipeline em: {device} | Dispositivo: {gpu_name}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not build the patched trl.trainer.grpo_trainer, so training will use trl's own trainer instead: RuntimeError: Direct module loading failed for UnslothGRPOTrainer: Unexpected optimization option triton.enable_persistent_tma_matmul, known options are ['TYPE_CHECKING', 'enable_auto_functionalized_v2', 'debug', 'disable_progress', 'verbose_progress', 'fx_graph_cache', 'fx_graph_remote_cache', 'autotune_local_cache', 'autotune_remote_cache', 'force_disable_caches', 'sleep_sec_TESTING_ONLY', 'custom_op_default_layout_constraint', 'cpp_wrapper', 'abi_compatible', 'c_shim_version', 'dce', 'static_weight_shapes', 'size_asserts', 'nan_asserts', 'pick_loop_orders', 'inplace_buffers', 'allow_buffer_reuse', 'memory_planning', 'memory_pool', 'benchmark_harness', 'epilogue_fusion', 'epilogue_fusion_first', 'pattern_matcher', 'b2b_gemm_pass', 'post_grad_custom_pre_pass', 'post_grad_custom_post_pass', 'joint_custom_pre_pass', 'joint_custom_post_pass', 'pre_grad_c

Iniciando Pipeline em: cuda | Dispositivo: NVIDIA GeForce RTX 3060


/tmp/ipykernel_196613/4131812714.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import os
import google.generativeai as genai
from datasets import Dataset
import pandas as pd
import time
from sklearn.model_selection import train_test_split

def carregar_e_preparar_dados(caminho_csv: str) -> pd.DataFrame:
    df = pd.read_csv(caminho_csv, sep=";", encoding="latin1")
    df.columns = df.columns.str.strip().str.upper()


    df["LABEL_CATEGORY"] = "true"
    return df

def gerar_proposta_falsa(candidato: str, proposta_real: str) -> str:
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        return ""

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemini-1.5-flash")

    prompt = f"""Você é um gerador de dados sintéticos para treinamento de checagem de fatos.
Candidato: {candidato}
Proposta Real: {proposta_real}

Gere UMA proposta FALSA/DISTORCIDA que pareça ter sido dita pelo candidato "{candidato}", baseando-se na proposta real acima.
Aplique um exagero inviável, alteração de público-alvo ou inclusão de custos/regras absurdas.
Retorne APENAS o texto da proposta falsa."""

    try:
        response = model.generate_content(prompt)
        return response.text.strip() if response.text else ""
    except Exception as e:
        print(f"Erro ao gerar fake para {candidato}: {e}")
        return ""


df_real = carregar_e_preparar_dados("./com_propostas.csv")


falsas_registros = []
print("A gerar propostas falsas vinculadas aos candidatos...")

for _, row in df_real.iterrows():
    cand_nome = row.get("NM_CANDIDATO", row.get("NM_URNA_CANDIDATO", "Desconhecido"))
    prop_real = row.get("PROPOSTA", "")

    if pd.isna(prop_real) or not str(prop_real).strip():
        continue

    prop_falsa = gerar_proposta_falsa(str(cand_nome), str(prop_real))
    
    time.sleep(3)

    if prop_falsa:
        falsas_registros.append(
            {
                "DS_CARGO": row.get("DS_CARGO", ""),
                "NM_UE": row.get("NM_UE", ""),
                "SQ_CANDIDATO": row.get("SQ_CANDIDATO", ""),
                "NM_CANDIDATO": row.get("NM_CANDIDATO", cand_nome),
                "NM_URNA_CANDIDATO": row.get("NM_URNA_CANDIDATO", cand_nome),
                "PROPOSTA": prop_falsa,
                "LABEL_CATEGORY": "false",
            }
        )

df_falsas = pd.DataFrame(falsas_registros)
df_final = pd.concat([df_real, df_falsas], ignore_index=True)

# 4. Exportação dos ficheiros bem separados
print("A guardar os ficheiros CSV...")

# Ficheiro 1: Apenas as propostas falsas geradas
df_falsas.to_csv("dataset_apenas_falsas.csv", index=False, sep=";", encoding="utf-8-sig")

# Ficheiro 2: Base completa, mas agrupada por candidato para facilitar a comparação visual
df_final_agrupado = df_final.sort_values(by=["NM_CANDIDATO", "LABEL_CATEGORY"], ascending=[True, False])
df_final_agrupado.to_csv("dataset_completo_agrupado.csv", index=False, sep=";", encoding="utf-8-sig")

# Ficheiro 3: Base completa embaralhada (formato final ideal para treinar o modelo)
df_final_treino = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
df_final_treino.to_csv("dataset_treino_embaralhado.csv", index=False, sep=";", encoding="utf-8-sig")

print("Concluído! Ficheiros guardados na pasta atual.")

A gerar propostas falsas vinculadas aos candidatos...
A guardar os ficheiros CSV...
Concluído! Ficheiros guardados na pasta atual.


In [3]:
max_seq_length = 4096
lora_rank = 32         

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.9.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.68 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.6. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
# Template ajustado para incluir a relação Candidato + Proposta
prompt_template = """### Instrução:
Classifique a proposta política a seguir atribuída ao candidato {candidato} como 'verdadeira' (true) ou 'falsa' (false).

### Candidato:
{candidato}

### Proposta:
{proposta}

### Resposta:
{resposta}"""


def formatar_prompts(dataframe):
    textos = []
    for _, row in dataframe.iterrows():
        candidato = row["NM_CANDIDATO"]
        if pd.isna(candidato) or not str(candidato).strip():
            candidato = row["NM_URNA_CANDIDATO"]

        texto = (
            prompt_template.format(
                candidato=candidato,
                proposta=row["PROPOSTA"],
                resposta=row["LABEL_CATEGORY"],
            )
            + tokenizer.eos_token
        )
        textos.append(texto)
    return pd.DataFrame({"text": textos})


# Divisão de Treino e Validação (80/20)
train_df, val_df = train_test_split(
    df_final, test_size=0.2, random_state=42, stratify=df_final["LABEL_CATEGORY"]
)

dataset_treino = Dataset.from_pandas(formatar_prompts(train_df))
dataset_validacao = Dataset.from_pandas(formatar_prompts(val_df))

In [5]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_treino,
    eval_dataset = dataset_validacao,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 4,     
        warmup_ratio = 0.05,
        num_train_epochs = 4,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_propostas",
    ),
)

# Iniciar Treinamento SFT
trainer_stats = trainer.train()
print("Treinamento finalizado com sucesso!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: reducing dataset_num_proc 6 -> 4 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/329 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/83 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 329 | Num Epochs = 4 | Total steps = 84
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080 of 7,331,909,632 (1.14% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.389870
10,1.196015
15,1.090644
20,1.030333
25,0.997891
30,0.942838
35,0.830437
40,0.830244
45,0.737813
50,0.677140


Unsloth: Restored added_tokens_decoder metadata in outputs_propostas/checkpoint-84/tokenizer_config.json.


Treinamento finalizado com sucesso!


In [9]:
gguf_directory = "modelo_propostas_gguf_v3"
quantization_method = "q4_k_m" # Opções: "q4_k_m", "q8_0", "f16"
# pra rodar localmente ou em outras plataformas
print(f"Exportando modelo para GGUF em quantização {quantization_method}...")

model.save_pretrained_gguf(
    gguf_directory, 
    tokenizer, 
    quantization_method = quantization_method
)

print(f"Modelo GGUF salvo com sucesso na pasta: ./{gguf_directory}")

Exportando modelo para GGUF em quantização q4_k_m...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in modelo_propostas_gguf_v3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in modelo_propostas_gguf_v3.


Found HuggingFace hub cache directory: /home/cesar/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Successfully copied all 3 files from cache to `modelo_propostas_gguf_v3`
Checking cache directory for required files...




Unsloth: Copying 1 files from cache to `modelo_propostas_gguf_v3`: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]


Successfully copied all 1 files from cache to `modelo_propostas_gguf_v3`



Unsloth: Preparing safetensor model files: 100%|██████████| 3/3 [00:00<00:00, 34007.87it/s]




Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [02:08<00:00, 42.76s/it]


Unsloth: Merge process complete. Saved to `/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/modelo_propostas_gguf_v3`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11030-mix-5ff778e (app-b11030-mix-5ff778e-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['modelo_propostas_gguf_v3_gguf/mistral-7b-instruct-v0.3.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...